# Cardiac Nexus — ECG Explainability (Integrated Gradients)

## Objective

Attach explainability to the trained ECG deep learning model so predictions come with a clinically inspectable reason, not just a probability. This notebook loads the checkpoint saved by `01_first_ecg_model.ipynb`, then uses Captum's Integrated Gradients to compute a per-sample, per-lead, per-timestep attribution map and overlays it directly on the raw ECG waveform.

This is the second pillar of the project (deep learning model + explainable AI). It is meant to be run after the baseline notebook has produced `/content/cardio_nexus_ecg_mi_baseline.pt`.

> Scope: research evaluation only. Attribution maps show what the model attended to, not a verified clinical diagnosis.

## Method

Integrated Gradients (Sundararajan et al., 2017) attributes a prediction to each input feature by integrating the gradient of the model's output along a straight path from a baseline input to the actual input:

$$IG_i(x) = (x_i - x_i') \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha (x - x'))}{\partial x_i} \, d\alpha$$

For a 12-lead, 1000-sample ECG tensor, the baseline $x'$ is a zero signal (a flatline), and the attribution is computed independently for every one of the 12,000 input values. Captum approximates the integral with a Riemann sum over a fixed number of interpolation steps.

1. Load the trained model and its saved weights.
2. Rebuild the PTB-XL test split and dataset exactly as in the baseline notebook.
3. Run Integrated Gradients on a handful of test recordings (true positives, true negatives, and any false predictions worth inspecting).
4. Plot each lead's raw waveform with its attribution trace directly beneath it, so high-attribution regions can be checked against known ECG morphology (e.g. ST-segment elevation, T-wave inversion).

In [ ]:
!pip -q install wfdb "pandas==2.2.2" numpy scikit-learn matplotlib seaborn torch tqdm captum

In [ ]:
from pathlib import Path
import ast
import random
import zipfile

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset

from captum.attr import IntegratedGradients

SEED = 42
DATA_ROOT = Path("/content/data")
CHECKPOINT_PATH = Path("/content/cardio_nexus_ecg_mi_baseline.pt")
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
IG_STEPS = 64
NUM_EXAMPLES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Checkpoint not found. Run 01_first_ecg_model.ipynb in this same Colab session first "
        "so /content/cardio_nexus_ecg_mi_baseline.pt exists, or re-upload a saved checkpoint here."
    )

In [ ]:
# Rebuild the PTB-XL metadata and test split exactly as in the baseline notebook.
metadata_paths = list(DATA_ROOT.rglob("ptbxl_database.csv"))
if not metadata_paths:
    raise FileNotFoundError(
        "PTB-XL data not found under /content/data. Run 01_first_ecg_model.ipynb first in this "
        "session so the dataset is downloaded and extracted."
    )
PTBXL_DIR = metadata_paths[0].parent

metadata = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col=0)
scp = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
scp = scp[scp["diagnostic"] == 1]
diagnostic_map = scp["diagnostic_class"].dropna().to_dict()

def diagnostic_classes(raw_codes):
    codes = ast.literal_eval(raw_codes) if isinstance(raw_codes, str) else raw_codes
    return {diagnostic_map[code] for code in codes if code in diagnostic_map}

metadata["diagnostic_classes"] = metadata["scp_codes"].apply(diagnostic_classes)
metadata["label"] = metadata["diagnostic_classes"].apply(lambda classes: int("MI" in classes))
test_df = metadata[metadata["strat_fold"] == 10].copy().reset_index(drop=True)

def load_ecg(row):
    record_path = PTBXL_DIR / row["filename_lr"]
    signal, _ = wfdb.rdsamp(str(record_path))
    signal = signal.astype(np.float32).T  # [12 leads, 1000 samples]
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True) + 1e-6
    return (signal - mean) / std

print("Test recordings available:", len(test_df))

In [ ]:
class ECG1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1)).squeeze(-1)

# weights_only=True restricts unpickling to tensors/plain data, since this checkpoint
# only ever contains a state_dict and a metrics dict.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model = ECG1DCNN().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded checkpoint metrics:", checkpoint.get("metrics"))

In [ ]:
# Captum expects a model whose forward pass returns a score to attribute. We wrap the
# logit in a sigmoid so the attributed quantity is the predicted probability itself.
class ProbabilityWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        return torch.sigmoid(self.base_model(x)).unsqueeze(-1)

prob_model = ProbabilityWrapper(model).to(DEVICE).eval()
integrated_gradients = IntegratedGradients(prob_model)

@torch.no_grad()
def predict_proba(signal_tensor):
    return torch.sigmoid(model(signal_tensor.unsqueeze(0).to(DEVICE))).item()

def explain(index):
    row = test_df.iloc[index]
    signal = load_ecg(row)
    input_tensor = torch.tensor(signal, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad_()
    baseline = torch.zeros_like(input_tensor)

    attribution, convergence_delta = integrated_gradients.attribute(
        input_tensor,
        baselines=baseline,
        n_steps=IG_STEPS,
        return_convergence_delta=True,
    )

    probability = torch.sigmoid(model(input_tensor)).item()
    return {
        "row": row,
        "signal": signal,
        "attribution": attribution.squeeze(0).detach().cpu().numpy(),
        "probability": probability,
        "convergence_delta": convergence_delta.item(),
    }

print("Integrated Gradients ready. Convergence delta close to 0 indicates a reliable attribution.")

In [ ]:
# Pick a mix of confidently-correct MI and non-MI examples so the attribution maps can be
# sanity-checked against known ECG morphology for both classes.
mi_candidates = test_df[test_df["label"] == 1].index.tolist()
non_mi_candidates = test_df[test_df["label"] == 0].index.tolist()
random.shuffle(mi_candidates)
random.shuffle(non_mi_candidates)

example_indices = mi_candidates[: NUM_EXAMPLES // 2] + non_mi_candidates[: NUM_EXAMPLES - NUM_EXAMPLES // 2]
explanations = [explain(i) for i in example_indices]

for result in explanations:
    label = "MI" if result["row"]["label"] == 1 else "non-MI"
    print(
        f"ecg_id={result['row'].name} true={label} "
        f"predicted_probability={result['probability']:.3f} "
        f"convergence_delta={result['convergence_delta']:.4f}"
    )

In [ ]:
def plot_explanation(result):
    signal = result["signal"]
    attribution = result["attribution"]
    label = "MI" if result["row"]["label"] == 1 else "non-MI"

    fig, axes = plt.subplots(12, 1, figsize=(12, 18), sharex=True)
    fig.suptitle(
        f"ecg_id={result['row'].name} | true={label} | predicted P(MI)={result['probability']:.3f}",
        y=1.0,
    )
    time_axis = np.arange(signal.shape[1]) / 100.0  # 100 Hz sampling rate

    max_abs_attr = np.abs(attribution).max() + 1e-8
    for lead_index, lead_name in enumerate(LEAD_NAMES):
        ax = axes[lead_index]
        ax.plot(time_axis, signal[lead_index], color="black", linewidth=0.8)
        normalized_attr = attribution[lead_index] / max_abs_attr
        ax.fill_between(
            time_axis,
            0,
            normalized_attr * signal[lead_index].std() * 3,
            where=normalized_attr >= 0,
            color="crimson",
            alpha=0.4,
            linewidth=0,
        )
        ax.fill_between(
            time_axis,
            0,
            normalized_attr * signal[lead_index].std() * 3,
            where=normalized_attr < 0,
            color="steelblue",
            alpha=0.4,
            linewidth=0,
        )
        ax.set_ylabel(lead_name, rotation=0, ha="right", va="center")
        ax.set_yticks([])
    axes[-1].set_xlabel("Time (s)")
    fig.text(
        0.01, 0.0,
        "Red = pushed prediction toward MI, Blue = pushed prediction toward non-MI "
        "(Integrated Gradients attribution, shaded relative to each lead's own scale)",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

for result in explanations:
    plot_explanation(result)

## Interpretation and limitations

Attribution maps show which parts of the input signal most influenced this specific model's specific prediction — they are a property of the model, not a verified clinical finding. A useful sanity check is whether high-attribution regions for MI-labeled recordings line up with clinically expected patterns (e.g. ST-segment or T-wave changes in the leads corresponding to the affected myocardial territory). A large `convergence_delta` for any example indicates the attribution approximation was less reliable for that sample and should be treated with extra caution.

This notebook explains the binary MI baseline model. The same `IntegratedGradients` wrapper pattern applies to the multi-label model in `02_multilabel_ecg_model.ipynb` by attributing each output class independently (Captum supports a `target` index for exactly this).